# Unit 2: Cryptography — Shor's Algorithm

**Companion notebook for *Quantum Computing From the Problem Down***

We factor $N = 15$ using Shor's algorithm, running the quantum period-finding subroutine on a cloud [Quokka](https://www.quokkacomputing.com/).

**What you'll see:**
1. The classical reduction: factoring → period-finding
2. Build the quantum period-finding circuit as QASM
3. Run it on a Quokka, extract the period from the measurement
4. Use the period to factor $N$

In [ ]:
import json
import math
from fractions import Fraction

import numpy as np
import matplotlib.pyplot as plt
import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

# --- Quokka connection ---
QUOKKA = "quokka1"
QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"

def _decode_quokka_counts(payload: dict) -> dict:
    """Normalize Quokka responses to a simple {bitstring: count} mapping."""
    if isinstance(payload, dict) and all(isinstance(v, int) for v in payload.values()):
        return payload

    if not isinstance(payload, dict):
        raise TypeError(f"Unexpected Quokka payload type: {type(payload)!r}")

    if payload.get("error_code", 0) not in (0, None):
        raise RuntimeError(f"Quokka error {payload.get('error_code')}: {payload.get('error')}")

    result = payload.get("result")
    if not isinstance(result, dict) or "c" not in result:
        raise ValueError(f"Unexpected Quokka result format: {payload}")

    counts = {}
    for shot in result["c"]:
        bitstring = ''.join(str(int(bit)) for bit in shot)
        counts[bitstring] = counts.get(bitstring, 0) + 1
    return counts

def run_qasm(program: str, shots: int = 1024) -> dict:
    result = requests.post(QUOKKA_URL, json={"script": program, "count": shots}, verify=False)
    result.raise_for_status()
    return _decode_quokka_counts(result.json())

print(f"Connected to {QUOKKA_URL}")

## 1. The classical part: factoring → period-finding

We want to factor $N = 15$. Pick $a = 7$ (coprime to 15).

Compute $f(x) = 7^x \bmod 15$ and look for the period:

In [ ]:
N = 15
a = 7

print(f"Factoring N = {N} using a = {a}")
print(f"\nf(x) = {a}^x mod {N}:")
print("-" * 20)

values = []
for x in range(16):
    fx = pow(a, x, N)
    values.append(fx)
    print(f"  f({x:2d}) = {fx}")

# Find the period classically
r_classical = next(x for x in range(1, N) if pow(a, x, N) == 1)
print(f"\nPeriod r = {r_classical}")

## 2. The quantum part: period-finding via QFT

We build a simplified circuit that demonstrates the quantum Fourier transform
extracting the period. For $N = 15$ with $a = 7$, the period is $r = 4$.

With a 4-qubit input register ($2^4 = 16$), the QFT should concentrate
measurements on multiples of $16/4 = 4$, i.e., outcomes $\{0, 4, 8, 12\}$.

In [ ]:
# Simplified QFT-based period-finding circuit for a=7, N=15
# This uses a compiled oracle specific to this problem
# (a general modular exponentiation circuit would be much larger)

qpe_circuit = """
OPENQASM 2.0;
include "qelib1.inc";

qreg q[4];   // input register (phase estimation)
creg c[4];   // measurement results

// Step 1: Superposition on all input qubits
h q[0];
h q[1];
h q[2];
h q[3];

// Step 2: Controlled modular exponentiation (compiled for a=7, N=15)
// For this specific case, the controlled-U^(2^k) operations
// amount to controlled phase rotations.
// The eigenvalue phases are multiples of 2*pi/4 = pi/2

// Controlled rotation encoding period r=4
// Phase = 2*pi * j/r for eigenvalue j
cu1(1.5707963) q[0], q[3];  // pi/2
cu1(3.1415927) q[1], q[3];  // pi
cu1(6.2831853) q[2], q[3];  // 2*pi = 0

// Step 3: Inverse QFT on input register
// QFT^{-1} on qubits 0,1,2
h q[0];
cu1(-1.5707963) q[0], q[1];
h q[1];
cu1(-0.7853982) q[0], q[2];
cu1(-1.5707963) q[1], q[2];
h q[2];

// Swap q[0] and q[2] (bit reversal)
cx q[0], q[2];
cx q[2], q[0];
cx q[0], q[2];

// Step 4: Measure input register
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
measure q[3] -> c[3];
"""

print(qpe_circuit)

In [ ]:
# Run on Quokka
results = run_qasm(qpe_circuit, shots=1024)

print("Measurement results:")
for outcome in sorted(results.keys()):
    count = results[outcome]
    # Convert binary string to integer (input register = first 3 qubits)
    k = int(outcome[:3], 2)  # first 3 bits are the input register
    print(f"  {outcome} → k={k:2d}, counts={count}")

## 3. Extract the period using continued fractions

In [ ]:
n_input_qubits = 3
Q = 2 ** n_input_qubits  # = 8

print(f"Extracting period from measurements (Q = {Q}):\n")

recovered_periods = []
for outcome in sorted(results.keys()):
    k = int(outcome[:n_input_qubits], 2)
    if k == 0:
        print(f"  k = {k}: no information (skip)")
        continue

    # k/Q ≈ j/r for some integer j
    frac = Fraction(k, Q).limit_denominator(N)
    r_candidate = frac.denominator

    # Verify: does a^r ≡ 1 (mod N)?
    valid = pow(a, r_candidate, N) == 1
    print(f"  k = {k}: k/Q = {k}/{Q} ≈ {frac} → r = {r_candidate} {'✓' if valid else '✗'}")

    if valid and r_candidate > 1:
        recovered_periods.append(r_candidate)

if recovered_periods:
    r = min(recovered_periods)  # take the smallest valid period
    print(f"\nRecovered period: r = {r}")
else:
    print("\nNo valid period found — would need to retry")
    r = r_classical  # fallback for demonstration

## 4. Factor N using the period

In [ ]:
print(f"N = {N}, a = {a}, r = {r}")
print()

if r % 2 != 0:
    print(f"r = {r} is odd — need to pick a different 'a' and retry")
else:
    x = pow(a, r // 2, N)
    print(f"a^(r/2) = {a}^{r//2} = {x} (mod {N})")

    if x == N - 1:
        print(f"a^(r/2) ≡ -1 (mod N) — need to pick a different 'a'")
    else:
        factor1 = math.gcd(x - 1, N)
        factor2 = math.gcd(x + 1, N)
        print(f"gcd({x}-1, {N}) = gcd({x-1}, {N}) = {factor1}")
        print(f"gcd({x}+1, {N}) = gcd({x+1}, {N}) = {factor2}")
        print()

        if factor1 != 1 and factor1 != N:
            print(f"★ {N} = {factor1} × {N // factor1}")
        elif factor2 != 1 and factor2 != N:
            print(f"★ {N} = {factor2} × {N // factor2}")
        else:
            print("Factoring failed — try a different 'a'")

In [ ]:
# Visualise the measurement distribution
labels = sorted(results.keys())
counts = [results[k] for k in labels]
k_values = [int(k[:n_input_qubits], 2) for k in labels]
is_peak = [k % (Q // r_classical) == 0 for k in k_values]
colors = ['#009688' if p else '#B0BEC5' for p in is_peak]

plt.figure(figsize=(10, 5))
plt.bar(labels, counts, color=colors)
plt.xlabel('Measurement outcome')
plt.ylabel('Counts')
plt.title(f'Shor\'s algorithm: period-finding for {a}^x mod {N}')
plt.xticks(rotation=45)
plt.legend(handles=[
    plt.Rectangle((0,0),1,1, color='#009688', label=f'QFT peaks (multiples of {Q//r_classical})'),
    plt.Rectangle((0,0),1,1, color='#B0BEC5', label='Other'),
])
plt.tight_layout()
plt.show()

## What's next?

- **QFT deep dive:** See the corresponding deep dive chapter
- **QPE deep dive:** See the corresponding deep dive chapter
- **The next unit:** [Unit 3 — Drug Discovery](../manuscript/03-drug-discovery.md) — where VQE simulates molecular interactions